# Evaluation, health domain (Kaggle)

Runs every metric of `docs/evaluation_metrics.md` for the **health** domain: 5 single-question
metrics, 2 multi-question metrics, latency, and the end-to-end check through OCR. Each is
measured for **no privacy** (standard RAG) and **ours** (Agents 1-3 + the session monitor).
Step-by-step guide: `docs/evaluation_metrics.md`, section "Running it".

**Before running** (right sidebar):
* **Session options**: Accelerator **GPU T4 x2**, Internet **On**.
* **Add-ons -> Secrets**: `GEMINI_API_KEY` (several keys, comma-separated, are used in turn when one runs out of daily requests). Tick each as attached to this notebook.

**To let it run on its own** (recommended): with `SCALE = "5h"` in cell 1, click **Save Version
-> Save & Run All (Commit) -> Save** (top right) and close Kaggle. Every cell runs in the
background; when it finishes, the version's **Output** tab holds `eval_results_health_5h.zip`.

| Cell | What it does | Time at `SCALE = "5h"` |
|---|---|---|
| 1 Setup | clone, install, Ollama and the models, keys | ~15 min (every session) |
| 2 Data | download the real data, copy the generated corpus, write the questions | ~3 min |
| 3 OCR | read 25 real scans (end-to-end check) | ~20 min |
| 4 Labels | Gemini marks private values and writes questions for the 25 scans | ~5 min |
| 5 Build | Agents 1-3 over ~350 documents (150 generated, 150 real, 2 x 25 scan parts) | ~2 h |
| 6 Single | 100 Target Info, 100 repeat, 200 membership and 100 utility questions | ~1 h |
| 7 Multi | 60 attack sessions (20 targets x 3 protocols) x 8 rounds | ~45 min |
| 8 End to end | the scans' questions on both stores | ~10 min |
| 9 Report | every number, REPORT.md and the figures | ~3 min |
| 10 Pack | `eval_results_health_5h.zip` to download and commit | ~1 min |

`SCALE = "5h"` (the default) fits in about 4.5 hours. Optionally run `SCALE = "quick"` first
(about 10 minutes, a few items per step) to check the set-up; `SCALE = "full"` is the plan's
full volumes (12-15 hours). Each scale keeps its results in its own folder. Run interactively,
a session stops when the tab closes: run cell 1 again and carry on from the first unfinished
cell, since every step skips what is already on disk.

In [ ]:
# 1 Setup (run at the start of every session)
SCALE = "5h"   # "5h": every metric on less data, ~4.5 h; "quick": ~10 min check; "full": the plan's volumes (~12-15 h)
MODEL = "llama3:8b"                       # answers, both pipelines (as the dashboard)
GEMINI_MODEL = "gemini-3.5-flash-lite,gemini-3.1-flash-lite"   # Agents 1-3 (fallback order)
OCR_MODEL = "qwen3-vl:8b-instruct"        # reads the scans (end-to-end check)
BRANCH = "main"

import os, subprocess, sys
from pathlib import Path

W = Path("/kaggle/working")
REPO = W / "3rd_sem_el"
os.environ["SANITISE_WORKDIR"] = "/tmp/sanitise"      # models stay out of /kaggle/working
os.environ["OLLAMA_NUM_PARALLEL"] = "4"   # answers are asked 2 questions x 2 pipelines at once
os.environ["EVAL_CONCURRENCY"] = "2"
if REPO.exists():
    subprocess.run(["git", "-C", str(REPO), "pull", "--ff-only"], check=True)
else:
    subprocess.run(["git", "clone", "--depth", "1", "-b", BRANCH, "https://github.com/raksha-sendhil/3rd_sem_el.git", str(REPO)], check=True)
os.chdir(REPO)
!git log --oneline -1
!pip install -q -r requirements.txt
!pip install -q -e ".[retrieval]" --no-deps
!pip install -q "sentence-transformers>=3.0" "chromadb>=0.5" matplotlib
sys.path.insert(0, str(REPO / "tools"))
import colab_cache
colab_cache.install_ollama_and_model(MODEL)
colab_cache.install_ollama_and_model(OCR_MODEL)
colab_cache.start_ollama()

from kaggle_secrets import UserSecretsClient
try:
    key = UserSecretsClient().get_secret("GEMINI_API_KEY")
except Exception as exc:
    raise RuntimeError("Add the Kaggle secret GEMINI_API_KEY (Add-ons -> Secrets), tick it as "
                       "attached to this notebook, and run this cell again.") from exc
WORK = W / "eval" / ("health" if SCALE == "full" else f"health_{SCALE}")
os.environ.update({
    "SANITISE_LLM_BACKEND": "ollama", "SANITISE_MODEL": MODEL, "SANITISE_LLM_TIMEOUT_S": "600",
    "SANITISE_RETRIEVAL_MODE": "hybrid", "SANITISE_OCR_MODEL": OCR_MODEL,
    "SANITISE_AGENTS_BACKEND": "gemini", "SANITISE_AGENTS_MODEL": GEMINI_MODEL,
    "SANITISE_GEMINI_API_KEY": key, "GEMINI_API_KEYS": key,
    "SANITISE_MONITOR_DB_PATH": str(WORK / "stores" / "monitor.sqlite"),
})
RUN = f"python tools/eval_suite/run.py {{step}} --domain health --work {WORK} --scale {SCALE}"
def run(step, extra=""):
    cmd = RUN.format(step=step) + " " + extra
    print("$", cmd, flush=True)
    subprocess.run(cmd, shell=True, check=False)
print(f"ready: answers {MODEL} | Agents 1-3 {GEMINI_MODEL} ({len(key.split(','))} key(s)) | "
      f"OCR {OCR_MODEL} | results in {WORK}")

In [ ]:
# 2 Data: real public text downloaded now, the generated corpus copied, every question written
run("data")

In [ ]:
# 3 End to end, part 1: the 50 real scans read by OCR (main's step 0), cut into parts
run("e2e-prep")

In [ ]:
# 4 End to end, part 2: Gemini marks each scan's private values and writes 2 questions after them
# and 2 about the content (saved in e2e/labels.json: open it and check a few, see the guide)
run("e2e-label")

In [ ]:
# 5 Build: Agents 1-3 over the generated corpus, the real corpus and both end-to-end sets
# (long; re-run after a time limit: documents already in a store are skipped)
run("build")

In [ ]:
# 6 Single-question attacks: Target Info, Repeat / ROUGE Prompt, membership inference, utility
run("single")

In [ ]:
# 7 Multi-question attacks: 180 sessions of 8 rounds, no privacy vs ours (with the monitor)
run("multi")

In [ ]:
# 8 End to end, part 3: the scans' questions on the gold-text store (A) and the OCR store (B)
run("e2e")

In [ ]:
# 9 Report: every metric, REPORT.md and the figures
run("report")
from IPython.display import Image, Markdown, display
display(Markdown((WORK / "results" / "REPORT.md").read_text()))
for fig in sorted((WORK / "results" / "figures").glob("*.png")):
    display(Image(str(fig)))

In [ ]:
# 10 Pack the results to download (Output tab, or Save Version) and commit to the repo
import shutil, zipfile
zip_path = W / f"eval_results_health{'' if SCALE == 'full' else '_' + SCALE}.zip"
keep = ["results", "answers", "data/targeted.json", "data/sessions.json", "data/repeat.json",
        "data/membership.json", "data/utility.json", "e2e/labels.json", "e2e/manifest.json"]
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
    for item in keep:
        p = WORK / item
        for f in ([p] if p.is_file() else sorted(p.rglob("*")) if p.exists() else []):
            if f.is_file():
                z.write(f, f"health/" + str(f.relative_to(WORK)))
print(f"{zip_path}: {zip_path.stat().st_size / 1e6:.1f} MB")
print("Download it from the Output tab (or Save Version), then follow 'Saving the results' in the guide.")